In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d
from render import N_POOLS, render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap
from parameter import (P, CellGeometry, CellProfile, CellMarker, Detector,
                       BlobNoise, ClusterNoise, NetworkNoise, FibreNoise, SheetNoise,
                       get_all_parameters)

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED = None
N_CAND = 1500
SIZE = 201
TILE = 256
K = 20

# lowest SH degree. 0 = pure size (absorbed by the volume normalisation),
# 1 = shifts the centroid off the seed. 2 = lowest true shape mode.
L_MIN = 2
# 4 is the ceiling of the hardcoded Cartesian forms
L = 4

# microns per LATERAL voxel
UM_PER_VOX = 0.325
# z voxels this many times COARSER than lateral
Z_RATIO = 1.0
# (sz, sy, sx) = voxel SIZE per axis, in lateral units
SPACING = (Z_RATIO, 1.0, 1.0)   
VOL = (128, 128, 128)
IMG = (128, 128, 3)

GEOM = CellGeometry()
DETECTOR = Detector()

# 4) DRAW THE TAPE

tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)
tape.drawSensor(shape=IMG)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, GEOM, size=VOL, spacing=SPACING, i=0, L=L, l_min=L_MIN)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, geom = GEOM)
fig = plot_surface_xyz_html(cell = cell, geom = GEOM,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:

PROFILE = CellProfile(
    Geometry = GEOM,
    Markers = dict(

        r = CellMarker(name="r", fluorophore="APC", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.80, s=1.40, mu= .55, width=.45, sharp=5.0,
                             scale=.35, clust=2.00, fill=.22, soft=.25),
                FibreNoise  (w=.30, s=1.30, mu= .20, width=.70, sharp=4.0,
                             lam=.25, length=6.0),
                NetworkNoise(w=.40, s=1.30, mu= .50, width=.55, sharp=3.5,
                             scale=.80, coherence=.40),
            ]),

        g = CellMarker(name="g", fluorophore="FITC", amp=2.0, polarity=0.2,
            noise_components=[
                BlobNoise   (w=.50, s=1.50, mu= .10, width=.60, sharp=7.5,
                             scale=.45),
                SheetNoise  (w=.60, s=1.50, mu= .40, width=1.20, sharp=6.0,
                             lam=1.90, coherence=.35, length=6.0),
                NetworkNoise(w=.90, s=1.40, mu= .20, width=1.10, sharp=7.0,
                             scale=1.00, coherence=.60),
            ]),

        b = CellMarker(name="b", fluorophore="PE", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.70, s=1.20, mu=-.55, width=.60, sharp=4.0,
                             scale=.30, clust=1.40, fill=.35, soft=.30),
                BlobNoise   (w=.45, s=1.30, mu=-.60, width=.70, sharp=3.0,
                             scale=.40),
                NetworkNoise(w=.35, s=1.20, mu=-.50, width=.90, sharp=7.0,
                             scale=.70, coherence=.30),
            ]),
    )
)

# edge_softness=0 -> clipped EXACTLY at the plasma membrane. All the softness in the final
# image is made by psf_project and detector, not baked into the object.
img = render_image(tape, PROFILE, cell, spacing=SPACING, um_per_vox=UM_PER_VOX)

rgb = to_rgb(cell["cell"], img)

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = GEOM.ELONG.v,
                        polar_deg = GEOM.POLAR_DEG.v, azim_deg = GEOM.AZIM_DEG.v, roll_deg = GEOM.ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
import psfmodels as psfm
import numpy as np
from scipy.signal import fftconvolve
from dataclasses import dataclass

In [ ]:
# Verify
FLUOROPHORES = {
    "DAPI": 0.461, "FITC": 0.519, "PE": 0.578, "APC": 0.660
}

# Optics: 
@dataclass(frozen=True)
class Optics:
    """Known Physical properties of the detector (Macsima) and the experiment."""
    um_per_px: float = 0.325 
    um_per_pz: float = 0.325 
    
    focal_um: float = 0.0
    
    # VERIFY
    # Numerical Aperture (NA)
    na: float = 0.45 #or 0.75
     
    wavelength_um: float = 0.530 # fallback
    
    # different refractive index of tissue and medium. VERIFY
    n_immersion: float = 1.0
    n_sample: float = 1.33
    
    # Thickness of the section
    section_um: float = 4.
    # Depth of the section CENTRE below the coverslip
    depth_um: float = 2.0
    
    @property
    def sample_depth_um(self):
        """Depth of the section centre below the coverslip."""
        return max(self.depth_um, self.section_um / 2)
    
    @property
    def tan_theta(self):
        return float(np.tan(np.arcsin(np.clip(self.na / self.n_immersion, 0.0, 0.999))))

optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
_KERNEL_CACHE = {}

def plane_heights(nz, um_per_pz):
    """Axial coordinate of each slice of a centred volume, in microns."""
    return (np.arange(nz) - (nz - 1) / 2.0) * um_per_pz

def psf_support_px(opt, thickness_um, pad=6):
    """Odd kernel width that contains the most defocused PSF for a section of `thickness_um`."""
    r = 0.5 * thickness_um * opt.tan_theta / opt.um_per_px
    return int(2 * (int(np.ceil(r)) + pad) + 1)

def quad_weights(nz):
    """Trapezoid weights for the depth integral -- half weight on the two cut faces."""
    w = np.ones(nz, float)
    w[0] = w[-1] = 0.5
    return w

def psf_kernels(opt, nxy, wavelength_um, z_um):
    
    z_um = np.asarray(z_um, float)
    # depth below the coverslip, per plane
    depth = opt.sample_depth_um + z_um
    zf = opt.sample_depth_um + opt.focal_um

    # psfmodels requires pz >= 0. Reachable in normal use: kryostat(centre_um=c) puts the
    # slab at depth in [c, c + section_um], so ANY negative centre_um trips this.
    if depth.min() < 0:
        need = float(opt.sample_depth_um - depth.min())
        raise ValueError(
            f"plane at depth {depth.min():+.3f} um sits above the coverslip; psfmodels needs "
            f"pz >= 0. Use Optics(depth_um={need:.2f}) or more (currently "
            f"{opt.sample_depth_um:.2f}), or section closer to the centre.")
    
    # Cache the kernel per fluorophore
    key = (nxy, round(float(wavelength_um), 6), round(float(zf), 6),
           opt.um_per_px, opt.na, opt.n_immersion, opt.n_sample, z_um.tobytes())
    if key in _KERNEL_CACHE:
        return _KERNEL_CACHE[key]
    
    base = dict(nx=int(nxy), dxy=opt.um_per_px, NA=opt.na, wvl=float(wavelength_um),
                    ni=opt.n_immersion, ni0=opt.n_immersion, ns=opt.n_sample)

    ks = [np.asarray(psfm.make_psf(z=[float(zf)], pz=float(d), model="scalar", **base)[0],
                            np.float32) for d in depth]
    
    ks = [k / (k.sum() + 1e-30) for k in ks]
    _KERNEL_CACHE[key] = ks
    return ks


def psf_project(slice_vol, z_um, optics, fluorophore = None):
    nz = slice_vol.shape[0]
    # the dye is a property of the marker, so the caller hands it over directly
    wavelength_um = optics.wavelength_um if fluorophore is None else FLUOROPHORES[fluorophore]
    w = quad_weights(nz)
    nxy = psf_support_px(opt = optics, thickness_um = float(np.ptp(z_um)) + optics.um_per_pz)
    ks = psf_kernels(opt = optics, nxy = nxy, wavelength_um = wavelength_um, z_um = z_um)
    out = np.zeros(slice_vol.shape[1:], np.float32)
    for i in range(nz):
        if w[i] == 0:
            continue
        out += np.float32(w[i]) * fftconvolve(slice_vol[i].astype(np.float32), ks[i], mode="same")
    return np.maximum(out, 0.0) / np.float32(w.sum())

def kryostat(vol, opt, centre_um = 0.0):
    """Cut a `section_um`-thick slab. Returns (subvolume, its plane heights in um)."""
    z = plane_heights(vol.shape[0], opt.um_per_pz)
    keep = np.abs(z - centre_um) <= opt.section_um / 2
    return vol[keep], z[keep]

def NormalizeData(stack, pct=99.5):
    return np.clip(stack / (np.percentile(stack, pct) + 1e-12), 0.0, 1.0)

subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, PROFILE.Markers[name].fluorophore)
                    for name, (v, z) in subs.items()], -1)

In [ ]:
def mask_collapse(mask, z_um, opt=None, mask_pct=0.95):
    """The honest 2D footprint of a 3D object seen through the exact optics.
    """
    cov = psf_project(slice_vol = mask.astype(np.float32), 
                      z_um = z_um, optics = opt)
    flat = np.sort(cov.ravel())[::-1]
    csum = np.cumsum(flat)
    if csum[-1] <= 0:
        return np.zeros(cov.shape, bool), cov
    k = int(np.searchsorted(csum, mask_pct * csum[-1]))
    thr = flat[min(k, flat.size - 1)]
    return cov >= thr, cov

sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
# VERIFY
AUTOFLUOR = {"DAPI": 0.20, "FITC": 0.60, "PE": 0.30, "APC": 0.10}


def _smooth_unit(w, sigma_px):
    """Circular Gaussian low-pass of a frozen field, rescaled to zero mean and unit variance.
    """
    ny, nx = w.shape
    fy = np.fft.fftfreq(ny)[:, None]
    fx = np.fft.rfftfreq(nx)[None, :]
    H = np.exp(-2 * np.pi ** 2 * float(sigma_px) ** 2 * (fy ** 2 + fx ** 2))
    z = np.fft.irfftn(np.fft.rfftn(w) * H, s=(ny, nx), axes=(0, 1))
    z = z - z.mean()
    return (z / (z.std() + 1e-12)).astype(np.float32)


def illumination(shape, illum_cv, vignette, tape):
    """Multiplicative flat-field: a smooth random component times a radial falloff."""
    ny, nx = shape
    rnd = 1.0 + illum_cv * _smooth_unit(tape["w_ill"], 0.25 * max(ny, nx))
    yy = (np.arange(ny) - (ny - 1) / 2.0)[:, None] / max(ny / 2.0, 1)
    xx = (np.arange(nx) - (nx - 1) / 2.0)[None, :] / max(nx / 2.0, 1)
    rr2 = np.clip((yy ** 2 + xx ** 2) / 2.0, 0.0, 1.0)     # 0 at centre, 1 at the corners
    return np.maximum(rnd * (1.0 - vignette * rr2), 0.0).astype(np.float32)


def detector(img_psf, profile, det, opt, tape, quantise=False,
             autofluor = AUTOFLUOR):
    """Turn a clean PSF projection (H, W, C) into a detector frame.
    """
    e_per_unit  = det.E_PER_UNIT.v
    read_e      = det.READ_E.v
    dark_e      = det.DARK_E.v
    adu_per_e   = det.ADU_PER_E.v
    offset_adu  = det.OFFSET_ADU.v
    af_scale_um = det.AF_SCALE_UM.v
    af_cv       = det.AF_CV.v
    illum_cv    = det.ILLUM_CV.v
    vignette    = det.VIGNETTE.v
    bit_depth   = det.BIT_DEPTH
    
    img_psf = np.asarray(img_psf, np.float32)
    ny, nx, nc = img_psf.shape
    if tape.z_shot.shape != (nc, ny, nx):
        raise ValueError(
            f"sensor tape is {tape.z_shot.shape} but the image is {(nc, ny, nx)}. "
            f"Re-run tape.drawSensor(shape=({ny}, {nx}, {nc})) -- and set IMG to match.")
    
    ill = illumination((ny, nx), illum_cv, vignette, tape)
    sig_px = max(af_scale_um / opt.um_per_px, 0.5)

    adu, diag = np.empty_like(img_psf), []
    for k, m in enumerate(markers):
        fl = profile.Markers[m].fluorophore
        af_level = float(autofluor.get(fl, 0.0))
        # autofluorescence is emitted BY the tissue, so it is illuminated like everything else
        af = af_level * np.maximum(1.0 + af_cv * _smooth_unit(tape["w_af"][k], sig_px), 0.0)

        # marker photoelectrons
        sig_e = e_per_unit * img_psf[..., k] * ill
        # autofluorescence photoelectrons
        bg_e = e_per_unit * af * ill
        e = np.maximum(sig_e + bg_e, 0.0)
        # shot noise scales as sqrt(signal): bright pixels are noisier
        noisy = e + np.sqrt(e) * tape["z_shot"][k] + read_e * tape["z_read"][k] + dark_e
        a = np.clip(noisy * adu_per_e + offset_adu, 0.0, 2 ** bit_depth - 1)
        adu[..., k] = np.rint(a) if quantise else a

        pk = float(np.percentile(sig_e, 99.9))
        bg = float(bg_e.mean())
        diag.append((m, fl, pk, bg, pk / (np.sqrt(pk + bg + read_e ** 2) + 1e-12)))
    return adu


markers = list(subs.keys())
img_adu = detector(img_psf, PROFILE, DETECTOR, optics, tape)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {PROFILE.Markers[markers[0]].fluorophore}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")